# Personalizar el modelo de IA del equipo (Fase 2) — Google Colab

Este notebook **entrena (fine-tuning con LoRA)** un modelo chico para que actúe como el
integrante IA del equipo dentro del diagramador UML, y lo exporta a **GGUF** para que corra
**localmente en el celular** (llama.cpp vía `llamadart`, ver `mobile/`).

| Paso | Qué hace |
|---|---|
| 1 | Instala Unsloth (entrenamiento LoRA rápido en la GPU T4 gratuita de Colab) |
| 2 | Carga el dataset propio del proyecto (`training/data/train.jsonl`, generado por `build_dataset.py`) |
| 3 | Mide el **modelo base** (línea de base) |
| 4 | Entrena con LoRA |
| 5 | Mide el **modelo personalizado** con las mismas métricas |
| 6 | Exporta a GGUF (Q4_K_M, ~0.8 GB) |
| 7 | Lo publica en el backend (`PUT /models/assistant`) → los celulares lo descargan |
| 8 | (Opcional) Lo sirve desde Colab como API compatible con OpenAI → la web también puede usarlo (CU-17) |

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`.

El modelo base es **Gemma 3 1B IT** (el mismo que ya usa la app). Con un celular de gama media
es lo más grande que corre con fluidez; la personalización compensa el tamaño: aprende el
formato exacto de la app, las herramientas del diagramador, a dirigirse a cada integrante por
su nombre y respuestas de mentoría destiladas de un modelo más grande (Gemini).

**Si Colab se desconecta no se pierde el progreso:** el notebook guarda en tu Google Drive
(`MyDrive/uml_ia/`) la medición del modelo base, checkpoints del entrenamiento cada ~40 pasos,
el modelo entrenado y el `.gguf`. Al volver, ejecutá las celdas de arriba hacia abajo: cada
paso que ya está hecho en Drive se salta o se retoma donde quedó.


In [ ]:
# 1. Dependencias (2-4 minutos)
!pip install -q unsloth
!pip install -q --no-deps "trl>=0.15" "datasets>=3.0"

## 1b. Guardar el progreso en Google Drive

Pide permiso para usar tu Drive (una vez por sesión). Todo lo importante se guarda en
`MyDrive/uml_ia/`, así un reinicio o desconexión de Colab no borra lo hecho.


In [ ]:
from google.colab import drive
import os, json
drive.mount("/content/drive")
SAVE = "/content/drive/MyDrive/uml_ia"
CKPT_DIR = f"{SAVE}/checkpoints"        # checkpoints del entrenamiento (para retomar)
ADAPTER_DIR = f"{SAVE}/modelo_entrenado" # LoRA final (si existe, no se reentrena)
os.makedirs(CKPT_DIR, exist_ok=True)

def save_json(name, obj):
    with open(f"{SAVE}/{name}", "w") as f: json.dump(obj, f)
def load_json(name):
    path = f"{SAVE}/{name}"
    return json.load(open(path)) if os.path.exists(path) else None

print("Guardado en Drive:", sorted(os.listdir(SAVE)))


## 2. Dataset del proyecto

Clona el repo para usar el mismo dataset y el mismo prompt que la app. Si el repo es privado,
subí a mano `training/data/train.jsonl` y `training/data/eval.jsonl` con el panel de archivos
de la izquierda (o poné un token de GitHub en `GITHUB_TOKEN`).

In [ ]:
import os, json, subprocess

REPO = "Luuuisss-o/parcial_sw"
BRANCH = "claude/exciting-tesla-sscl8u"   # cambiar a "main" cuando se mergee
GITHUB_TOKEN = ""                          # solo si el repo es privado

if not os.path.exists("parcial_sw"):
    auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, f"https://{auth}github.com/{REPO}.git"],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)

DATA = "parcial_sw/training/data" if os.path.exists("parcial_sw/training/data/train.jsonl") else "."
def load_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]
train_rows = load_jsonl(f"{DATA}/train.jsonl")
eval_rows = load_jsonl(f"{DATA}/eval.jsonl")
print(len(train_rows), "ejemplos de entrenamiento,", len(eval_rows), "de evaluación")
print(train_rows[0]["messages"][1]["content"][:300])

In [ ]:
# 3. Modelo base
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
import torch

BASE_MODEL = "unsloth/gemma-3-1b-it"
MAX_SEQ = 3072

model, tokenizer = FastModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ,
    load_in_4bit=False, load_in_8bit=False, full_finetuning=False,
)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")

### Métricas

Sobre los ejemplos de evaluación (que el modelo nunca ve al entrenar):

- **JSON válido**: la respuesta es exactamente el formato que la app sabe ejecutar.
- **Herramientas correctas**: las acciones propuestas son las esperadas (mismas tools, en orden).
- **Nombre**: la respuesta empieza dirigiéndose a quien preguntó.
- **Mentoría sin acciones**: en preguntas teóricas no propone tocar el diagrama.

In [ ]:
import re

def first_json(text):
    start = text.find("{")
    depth, in_str, esc = 0, False, False
    for i in range(max(start, 0), len(text)):
        ch = text[i]
        if in_str:
            if esc: esc = False
            elif ch == "\\": esc = True
            elif ch == '"': in_str = False
            continue
        if ch == '"': in_str = True
        elif ch == "{": depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0: return text[start:i+1]
    return None

def generate(user_content, max_new_tokens=512):
    prompt = tokenizer.apply_chat_template([{"role": "user", "content": user_content}],
                                           tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def evaluate(rows, n=60, show=3):
    FastModel.for_inference(model)
    stats = {"json": 0, "tools": 0, "name": 0, "mentor_ok": 0, "mentor_total": 0}
    rows = rows[:n]
    for i, r in enumerate(rows):
        user = r["messages"][0]["content"]
        want = json.loads(r["messages"][1]["content"])
        who = re.search(r"Quien pregunta: (\S+)", user).group(1)
        raw = generate(user)
        try:
            got = json.loads(first_json(raw) or "")
            stats["json"] += 1
        except Exception:
            got = {"say": raw, "actions": []}
        tools_want = [a["tool"] for a in want["actions"]]
        tools_got = [a.get("tool") for a in got.get("actions", []) if isinstance(a, dict)]
        stats["tools"] += tools_got == tools_want
        stats["name"] += str(got.get("say", "")).strip().startswith(who)
        if not tools_want:
            stats["mentor_total"] += 1
            stats["mentor_ok"] += not tools_got
        if i < show:
            print("PREGUNTA:", user.rsplit("Consulta actual de", 1)[1].strip()[:120])
            print("RESPUESTA:", raw[:300], "\n")
    m = {k: round(100 * stats[k] / len(rows), 1) for k in ("json", "tools", "name")}
    m["mentor_sin_acciones"] = round(100 * stats["mentor_ok"] / max(1, stats["mentor_total"]), 1)
    return m

# La línea de base se mide UNA vez (con el modelo sin entrenar) y queda en Drive.
baseline = load_json("baseline.json")
if baseline is None and os.path.isdir(ADAPTER_DIR):
    print("El modelo ya está entrenado y no hay línea de base guardada: se omite la comparación.")
elif baseline is None:
    baseline = evaluate(eval_rows)
    save_json("baseline.json", baseline)
print("MODELO BASE (%):", baseline)

In [ ]:
# 4. Entrenamiento con LoRA (≈ 20-40 min en una T4). Guarda checkpoints en Drive y,
#    si Colab se cortó a mitad, RETOMA desde el último en vez de empezar de cero.
import glob
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

if os.path.isdir(ADAPTER_DIR):
    print("Modelo ya entrenado en Drive -> se carga, no se reentrena:", ADAPTER_DIR)
    model, tokenizer = FastModel.from_pretrained(
        model_name=ADAPTER_DIR, max_seq_length=MAX_SEQ,
        load_in_4bit=False, load_in_8bit=False, full_finetuning=False,
    )
    tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")
else:
    model = FastModel.get_peft_model(
        model,
        finetune_vision_layers=False, finetune_language_layers=True,
        finetune_attention_modules=True, finetune_mlp_modules=True,
        r=16, lora_alpha=32, lora_dropout=0, bias="none", random_state=3407,
    )

    def to_text(row):
        return {"text": tokenizer.apply_chat_template(row["messages"], tokenize=False).removeprefix("<bos>")}

    train_ds = Dataset.from_list(train_rows).map(to_text)

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=train_ds,
        args=SFTConfig(
            dataset_text_field="text", max_seq_length=MAX_SEQ,
            per_device_train_batch_size=4, gradient_accumulation_steps=2,
            num_train_epochs=2, learning_rate=2e-4, warmup_steps=10,
            lr_scheduler_type="linear", optim="adamw_8bit", weight_decay=0.01,
            logging_steps=10, seed=3407, report_to="none",
            output_dir=CKPT_DIR, save_strategy="steps", save_steps=40, save_total_limit=2,
        ),
    )
    # Solo se aprende la RESPUESTA (el JSON), no a repetir el prompt.
    trainer = train_on_responses_only(trainer, instruction_part="<start_of_turn>user\n",
                                      response_part="<start_of_turn>model\n")
    resume = bool(glob.glob(f"{CKPT_DIR}/checkpoint-*"))
    print("Retomando desde el último checkpoint" if resume else "Entrenamiento desde cero")
    stats = trainer.train(resume_from_checkpoint=resume)
    print(stats)
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    print("Modelo entrenado guardado en Drive:", ADAPTER_DIR)


In [ ]:
# 5. Evaluación del modelo personalizado (mismas métricas, mismos ejemplos)
tuned = load_json("tuned.json")
if tuned is None:
    tuned = evaluate(eval_rows)
    save_json("tuned.json", tuned)
print("MODELO BASE          (%):", baseline)
print("MODELO PERSONALIZADO (%):", tuned)


In [ ]:
# 6. Exportar a GGUF para el celular (Q4_K_M ≈ 0.8 GB; si falla, Q8_0 ≈ 1.1 GB)
import glob
OUT = "gemma3-1b-uml-equipo"
try:
    model.save_pretrained_gguf(OUT, tokenizer, quantization_method="q4_k_m")
except Exception as e:
    print("q4_k_m no disponible en esta versión de Unsloth:", e, "\n-> exportando Q8_0")
    model.save_pretrained_gguf(OUT, quantization_type="Q8_0")
GGUF = sorted(glob.glob(f"{OUT}*/*.gguf") + glob.glob(f"{OUT}*.gguf"), key=os.path.getsize)[-1]
print(GGUF, round(os.path.getsize(GGUF) / 1e9, 2), "GB")

# Copia en Drive: sirve para subirlo después o bajarlo a la PC sin reexportar.
import shutil
shutil.copy(GGUF, f"{SAVE}/{os.path.basename(GGUF)}")
print("Copia en Drive:", f"{SAVE}/{os.path.basename(GGUF)}")

## 7. Publicar el modelo en el backend

El backend **solo aloja** el archivo (`PUT /models/assistant`, requiere un usuario con rol
`admin`); la inferencia corre en el celular. La app consulta `GET /models/assistant` y, si hay
una versión nueva, la descarga y verifica su SHA-256.

`API_URL` tiene que ser accesible desde Colab: el backend desplegado (EC2), o tu PC expuesta
con `cloudflared tunnel --url http://localhost:8000`. Si no, descargá el `.gguf` desde el panel
de archivos y subilo vos con el mismo endpoint (o copialo a `backend/data/models/`).

In [ ]:
import requests, datetime, getpass, glob
API_URL = "https://<tu-backend>"          # sin "/" final
ADMIN_EMAIL = "ad1@example.com"
VERSION = datetime.datetime.now().strftime("%Y.%m.%d-%H%M")

# Si la sesión se reinició después de exportar, usa lo guardado en Drive.
if "GGUF" not in globals():
    GGUF = sorted(glob.glob(f"{SAVE}/*.gguf"), key=os.path.getsize)[-1]
baseline, tuned = load_json("baseline.json"), load_json("tuned.json")

token = requests.post(f"{API_URL}/auth/sign-in",
                      json={"email": ADMIN_EMAIL, "password": getpass.getpass("Contraseña admin: ")}).json()["access_token"]
with open(GGUF, "rb") as f:
    r = requests.put(f"{API_URL}/models/assistant", headers={"Authorization": f"Bearer {token}"},
                     files={"file": (os.path.basename(GGUF), f, "application/octet-stream")},
                     data={"version": VERSION, "base_model": BASE_MODEL,
                           "notes": f"LoRA r=16, 2 épocas, {len(train_rows)} ejemplos. Base: {baseline}. Personalizado: {tuned}"},
                     timeout=3600)
print(r.status_code, r.json())

## 8. (Opcional) Servir el modelo personalizado desde Colab para la web (CU-17)

Expone el GGUF como **API compatible con OpenAI** con `llama-server` y un túnel público.
En el chat de la web, el dueño del diagrama elige *Modelo local / Colab*, pega la URL que
imprime el túnel + `/v1` y como modelo cualquier nombre (p. ej. `uml-equipo`).

El modelo de 1B entiende el formato JSON de la app; para la web (que usa herramientas nativas
y respuestas más largas) conviene un modelo más grande servido igual, p. ej. `qwen2.5:7b` con
Ollama en Colab.

In [ ]:
import subprocess, time, re
LLAMA_SERVER = (glob.glob("llama.cpp/llama-server") + glob.glob("llama.cpp/build/bin/llama-server") or [None])[0]
if LLAMA_SERVER is None:
    !pip install -q "llama-cpp-python[server]"
    server = subprocess.Popen(["python", "-m", "llama_cpp.server", "--model", GGUF, "--host", "0.0.0.0", "--port", "8080",
                               "--n_gpu_layers", "-1", "--chat_format", "gemma"])
else:
    server = subprocess.Popen([LLAMA_SERVER, "-m", GGUF, "--host", "0.0.0.0", "--port", "8080", "-ngl", "99"])
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8080"], stderr=subprocess.PIPE, text=True)
for line in tunnel.stderr:
    m = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
    if m:
        print("Endpoint para la web:", m.group(0) + "/v1")
        break